# Lab 11: Cascadia Slow Slip Inversion

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/12_cascadia_sse_inversion.ipynb)

## Introduction

In Lab 2 you used station ALBH to identify the 2015–2016 Cascadia slow slip event (SSE) in a GNSS residual time series. In this lab you will extend that analysis to a network of coastal Washington and BC stations and **invert for the spatial distribution of slip** on the Cascadia subduction interface using the real McCrory et al. triangulated plate interface geometry.

This is the inverse problem from the lecture: given observed surface displacements $\mathbf{d}$ and a Green's function matrix $\mathbf{G}$ computed from elastic dislocation theory, find the slip distribution $\mathbf{m}$ that best explains the data.

## Learning objectives

- extract SSE offsets from GNSS time series at multiple stations and propagate uncertainties
- load and filter a real triangulated Cascadia fault mesh
- build a Green's function matrix with cutde
- run a damped non-negative least-squares inversion
- choose $\lambda$ with the L-curve
- compute and interpret the resolution matrix

## Notebook outline
- [Setup](#setup)
- [Part I: Extract SSE offsets](#part-i-extract-sse-offsets)
- [Part II: Cascadia fault mesh](#part-ii-cascadia-fault-mesh)
- [Part III: Green's function matrix](#part-iii-greens-function-matrix)
- [Part IV: Invert for slip](#part-iv-invert-for-slip)
- [Part V: L-curve](#part-v-l-curve)
- [Part VI: Resolution matrix](#part-vi-resolution-matrix)
- [Synthesis](#synthesis)
- [Summary](#summary)

Archived GNSS data: [CRESCENT GNSS time-series dataset](https://zenodo.org/records/20518159)  
Fault geometry: [McCrory et al. Cascadia interface mesh](https://github.com/cascadiaquakes/comparing-interface-geomery)


In [ ]:
%pip install -q cutde h5netcdf bokeh-sampledata

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cutde.halfspace as hs
import requests, io
from scipy.optimize import nnls
from scipy.linalg import lstsq
from bokeh_sampledata.us_states import data as US_STATES

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.grid': True, 'grid.alpha': 0.2})

NU    = 0.25     # Poisson ratio
MU    = 30e9     # shear modulus (Pa)
SCALE = 111195.0 # m per degree lat
REF_LON, REF_LAT = -125.5, 48.0   # local Cartesian reference

def lonlat_to_m(lon, lat):
    x = (np.asarray(lon) - REF_LON) * SCALE * np.cos(np.radians(REF_LAT))
    y = (np.asarray(lat) - REF_LAT) * SCALE
    return x, y

print('Setup complete.')


## Part I: Extract SSE offsets

### 1a — Load the CRESCENT GNSS dataset

Same Zenodo NetCDF as Lab 2. Variables include `east_m`, `north_m`, `up_m` and
per-observation uncertainties `east_sigma_m`, `north_sigma_m`, `up_sigma_m`.
We use the diagonal uncertainty structure as the data covariance for weighting.


In [ ]:
GNSS_URL = "https://zenodo.org/records/20518159/files/gnss_SOPAC_trend_2010_2025.nc?download=1"
print("Downloading (~150 MB)...")
response = requests.get(GNSS_URL, timeout=240)
response.raise_for_status()
ds = xr.open_dataset(io.BytesIO(response.content), engine="h5netcdf")
print(f"Loaded: {ds.sizes['station']} stations, "
      f"{pd.to_datetime(ds.time.values).min().date()} to "
      f"{pd.to_datetime(ds.time.values).max().date()}")
print(f"Variables: {list(ds.data_vars)}")


### 1b — Select coastal stations above the 2015–2016 SSE zone

We target stations in 46–50°N where the SSE signal is expected to be strongest.
The code tries each station name and keeps those present in the dataset.


In [ ]:
TARGET_STATIONS = [
    "albh", "nano", "pgc5", "ptaa",    # BC / Vancouver Island
    "p401", "p435", "p160", "p403",    # Washington coast (north)
    "p441", "p162", "pabh",            # Washington coast (south)
    "sc02", "blyn", "seat",            # Puget Sound / inland
    "fisw", "lkcp",                    # Southern Washington
]

available = [s for s in TARGET_STATIONS if s in ds.station.values]
print(f"Found {len(available)}/{len(TARGET_STATIONS)} target stations:")
print("  ", ", ".join(available))

sta_data = {}
for sta in available:
    s = ds.sel(station=sta)
    sta_data[sta] = pd.DataFrame({
        "time":     pd.to_datetime(s.time.values),
        "dec_year": s.dec_year.values,
        "east_mm":  1000 * s.east_m.values,
        "north_mm": 1000 * s.north_m.values,
        "up_mm":    1000 * s.up_m.values,
        "se":       1000 * s.east_sigma_m.values,
        "sn":       1000 * s.north_sigma_m.values,
        "su":       1000 * s.up_sigma_m.values,
    })

print(f"\n{'Station':8s} {'Lon':>10s} {'Lat':>8s} {'N epochs':>10s}")
print("-" * 42)
for sta in available:
    s = ds.sel(station=sta)
    lon = float(s.lon.values) if 'lon' in s else float(s.longitude.values)
    lat = float(s.lat.values) if 'lat' in s else float(s.latitude.values)
    print(f"{sta:8s} {lon:10.3f} {lat:8.3f} {len(sta_data[sta]):>10d}")


### 1c — Fit trend + seasonal, extract SSE offsets with uncertainties

As in Lab 2: fit linear + annual + semiannual model per station, then measure
the SSE displacement as mean(residual post) − mean(residual pre).

Uncertainty propagation: if daily position noise is $\sigma_{day}$ and we
average over $n_{pre}$ and $n_{post}$ days,

$$\sigma_{offset} = \sigma_{day}\sqrt{\frac{1}{n_{post}} + \frac{1}{n_{pre}}}$$

The resulting $\sigma$ values populate the diagonal of the data covariance matrix
$\mathbf{C}_d$ used to weight the inversion.

The **2015–2016 SSE** ran 2015-12-28 to 2016-01-27 (Bartlow 2020 catalog).


In [ ]:
def build_design_matrix(t, seasonal=True):
    t0 = t.mean(); dt = t - t0
    cols = [np.ones_like(t), dt]
    if seasonal:
        for p in [1.0, 0.5]:
            omega = 2 * np.pi / p
            cols += [np.sin(omega * t), np.cos(omega * t)]
    return np.column_stack(cols)

def fit_residual(df, comp, sig_col):
    t, d, sig = df.dec_year.values, df[comp].values, df[sig_col].values
    G = build_design_matrix(t)
    W = np.diag(1.0 / sig**2)
    m, _, _, _ = lstsq(G.T @ W @ G, G.T @ W @ d)
    return d - G @ m

SSE_START   = pd.Timestamp("2015-12-28")
SSE_END     = pd.Timestamp("2016-01-27")
WINDOW_DAYS = 21

offsets = {}

for sta in available:
    df = sta_data[sta].copy()
    df['res_e'] = fit_residual(df, 'east_mm',  'se')
    df['res_n'] = fit_residual(df, 'north_mm', 'sn')

    pre  = df.time.between(SSE_START - pd.Timedelta(days=WINDOW_DAYS), SSE_START)
    post = df.time.between(SSE_END,   SSE_END   + pd.Timedelta(days=WINDOW_DAYS))

    if pre.sum() < 5 or post.sum() < 5:
        print(f"  {sta}: skipped (insufficient data in SSE windows)")
        continue

    de = df.loc[post, 'res_e'].mean() - df.loc[pre, 'res_e'].mean()
    dn = df.loc[post, 'res_n'].mean() - df.loc[pre, 'res_n'].mean()

    # Propagated uncertainty from daily sigmas (diagonal C_d)
    sig_e = df.loc[pre | post, 'se'].mean() * np.sqrt(1/post.sum() + 1/pre.sum())
    sig_n = df.loc[pre | post, 'sn'].mean() * np.sqrt(1/post.sum() + 1/pre.sum())

    s = ds.sel(station=sta)
    lon = float(s.lon.values) if 'lon' in s else float(s.longitude.values)
    lat = float(s.lat.values) if 'lat' in s else float(s.latitude.values)
    offsets[sta] = dict(de=de, dn=dn, se=sig_e, sn=sig_n, lon=lon, lat=lat)

print(f"SSE offsets for {len(offsets)} stations:")
print(f"{'Station':8s} {'dE(mm)':>9s} {'dN(mm)':>9s} {'se(mm)':>8s} {'sn(mm)':>8s}")
print("-" * 48)
for sta, v in offsets.items():
    print(f"{sta:8s} {v['de']:9.2f} {v['dn']:9.2f} {v['se']:8.3f} {v['sn']:8.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
for code in ('WA', 'OR', 'ID'):
    ax.plot(US_STATES[code]['lons'], US_STATES[code]['lats'], color='0.35', lw=0.8)

scale = 0.2  # degrees per mm
for sta, v in offsets.items():
    ax.quiver(v['lon'], v['lat'], v['de']*scale, v['dn']*scale,
              scale=1, scale_units='xy', angles='xy',
              color='steelblue', width=0.004, zorder=4)
    ax.text(v['lon']+0.05, v['lat']+0.05, sta.upper(), fontsize=7)

ax.quiver(-127.5, 46.3, 5*scale, 0, scale=1, scale_units='xy',
          angles='xy', color='k', width=0.005)
ax.text(-127.5, 46.1, '5 mm', fontsize=8)

ax.set_xlim(-129, -121); ax.set_ylim(45.5, 51)
ax.set_aspect(1/np.cos(np.radians(48)))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('2015–2016 Cascadia SSE — GNSS horizontal offsets\n'
             '(trend + seasonal removed, 21-day pre/post windows)')
plt.tight_layout(); plt.show()


> **Part I questions:**
> 1. What is the dominant direction of horizontal displacement? Is it consistent with slow slip on the Cascadia subduction interface?
> 2. Which station shows the largest displacement? Where is it relative to the subduction zone?
> 3. How large are the per-station uncertainties relative to the signal? Which stations have the best signal-to-noise ratio?
> 4. We use only east and north in the inversion. What additional information would the up component provide? Why is it sometimes excluded?


## Part II: Cascadia fault mesh

We use the **McCrory et al.** Cascadia plate interface, triangulated with Gmsh into 1,545 triangles.
We restrict the inversion to 46–50°N at 8–35 km depth — the northern Cascadia SSE zone —
giving **526 triangular patches**.

Mesh coordinates are in lon/lat/depth_km. We parse the Gmsh 4.1 format, filter to the SSE zone,
and convert to local Cartesian meters (required by cutde).


In [ ]:
MESH_URL = (
    "https://raw.githubusercontent.com/cascadiaquakes/"
    "comparing-interface-geomery/"
    "f6c1f2b58cc63fec8e3e9fe872198ebb6c2f59f9/"
    "generateMesh/mccrory/mccrory_ll.msh"
)
print("Downloading McCrory mesh...")
r = requests.get(MESH_URL, timeout=30)
r.raise_for_status()
mesh_lines = r.text.splitlines()
print(f"Downloaded: {len(mesh_lines)} lines")


def parse_gmsh4(lines):
    """Parse Gmsh 4.1 .msh file. Returns nodes dict and triangle list."""
    nodes, triangles = {}, []
    i = 0
    while i < len(lines):
        tok = lines[i].strip()
        if tok == '$Nodes':
            i += 1
            nb = int(lines[i].split()[0]); i += 1
            for _ in range(nb):
                n = int(lines[i].split()[3]); i += 1
                tags = [int(lines[i+j]) for j in range(n)]; i += n
                for j, tag in enumerate(tags):
                    nodes[tag] = list(map(float, lines[i+j].split()))[:3]
                i += n
        elif tok == '$Elements':
            i += 1
            nb = int(lines[i].split()[0]); i += 1
            for _ in range(nb):
                bp = lines[i].split(); etype = int(bp[2]); n = int(bp[3]); i += 1
                if etype == 2:
                    for j in range(n):
                        p = list(map(int, lines[i+j].split()))
                        triangles.append(p[1:4])
                i += n
        else:
            i += 1
    return nodes, triangles


nodes, tris = parse_gmsh4(mesh_lines)
tag2idx  = {k: i for i, k in enumerate(sorted(nodes.keys()))}
node_lld = np.array([[nodes[k][0], nodes[k][1], nodes[k][2]]
                      for k in sorted(nodes.keys())])   # lon, lat, dep_km
tri_tags = np.array([[tag2idx[t] for t in tri] for tri in tris])

cent_lon = node_lld[tri_tags, 0].mean(axis=1)
cent_lat = node_lld[tri_tags, 1].mean(axis=1)
cent_dep = node_lld[tri_tags, 2].mean(axis=1)   # negative = below surface

print(f"Full mesh: {len(nodes)} nodes, {len(tris)} triangles")
print(f"  Lat:   {cent_lat.min():.1f}–{cent_lat.max():.1f} °N")
print(f"  Depth: {cent_dep.min():.1f}–{cent_dep.max():.1f} km")

# Filter to SSE zone
sse_mask     = ((cent_lat > 46) & (cent_lat < 50) &
                (cent_dep > -35) & (cent_dep < -8))
sse_tri_tags = tri_tags[sse_mask]
N_PATCHES    = sse_mask.sum()
print(f"\nSSE zone (46–50°N, 8–35 km): {N_PATCHES} patches")

# Convert to local Cartesian meters for cutde
node_x, node_y = lonlat_to_m(node_lld[:, 0], node_lld[:, 1])
node_z   = node_lld[:, 2] * 1000   # km → m (negative)
node_xyz = np.column_stack([node_x, node_y, node_z])
tris_m   = node_xyz[sse_tri_tags]   # (N_PATCHES, 3, 3)

cent_lon_sse = cent_lon[sse_mask]
cent_lat_sse = cent_lat[sse_mask]
cent_dep_sse = cent_dep[sse_mask]

# Map the mesh
fig, ax = plt.subplots(figsize=(8, 7))
for code in ('WA', 'OR'):
    ax.plot(US_STATES[code]['lons'], US_STATES[code]['lats'], color='0.35', lw=0.8)
sc = ax.scatter(cent_lon_sse, cent_lat_sse, c=-cent_dep_sse,
                cmap='plasma', s=6, zorder=3)
plt.colorbar(sc, ax=ax, label='Depth (km)')
for sta, v in offsets.items():
    ax.plot(v['lon'], v['lat'], 'k^', ms=6, zorder=5)
    ax.text(v['lon']+0.05, v['lat']+0.05, sta.upper(), fontsize=7)
ax.set_xlim(-129, -121); ax.set_ylim(45.5, 51)
ax.set_aspect(1/np.cos(np.radians(48)))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'McCrory Cascadia mesh — SSE inversion zone\n'
             f'{N_PATCHES} patches, 46–50°N, 8–35 km')
plt.tight_layout(); plt.show()


> **Part II questions:**
> 1. Is the system overdetermined or underdetermined? ($N_{data} = 2 \times N_{stations}$, $N_{params}$ = `N_PATCHES`.)
> 2. Cascadia SSEs are typically at 25–40 km depth. Why include shallower patches in the inversion?
> 3. The mesh nodes are in lon/lat/km. Why do we convert to local Cartesian meters before calling cutde?


## Part III: Green's function matrix

$\mathbf{G}$ has shape $(M \times N)$ where $M = 2N_{stations}$ and $N = N_{patches}$.
Column $k$ is the surface displacement (E, N interleaved, in mm) from **1 m of thrust slip** on patch $k$.

Slip direction: `ds = +1` (reverse/thrust, hanging wall up-dip). For the Cascadia geometry
this produces westward and downward motion at coastal stations — consistent with observed SSE signals.

The data covariance $\mathbf{C}_d = \text{diag}(\sigma_i^2)$ from Part I feeds directly
into the weighted inversion in Part IV.


In [ ]:
sta_list = list(offsets.keys())
N_STA    = len(sta_list)
M_OBS    = 2 * N_STA

d_obs  = np.zeros(M_OBS)
sig_d  = np.zeros(M_OBS)
obs_xy = np.zeros((N_STA, 3))

for i, sta in enumerate(sta_list):
    v = offsets[sta]
    d_obs[2*i],   d_obs[2*i+1]   = v['de'], v['dn']
    sig_d[2*i],   sig_d[2*i+1]   = v['se'], v['sn']
    x, y = lonlat_to_m(v['lon'], v['lat'])
    obs_xy[i] = [x, y, 0.0]

print(f"Observations M = {M_OBS}  ({N_STA} stations × 2 components)")
print(f"Parameters   N = {N_PATCHES}  patches")
print(f"Ratio M/N      = {M_OBS/N_PATCHES:.3f}  (strongly underdetermined)")
print(f"sigma range    = {sig_d.min():.3f}–{sig_d.max():.3f} mm")

# Build G
print(f"\nBuilding G ({M_OBS}×{N_PATCHES})...")
G = np.zeros((M_OBS, N_PATCHES))
unit_slip = np.array([[0.0, 1.0, 0.0]])   # pure thrust (ds=+1)

for k in range(N_PATCHES):
    disp_k     = hs.disp_free(obs_xy, tris_m[k:k+1], unit_slip, nu=NU)
    G[0::2, k] = disp_k[:, 0] * 1000   # East  mm/m
    G[1::2, k] = disp_k[:, 1] * 1000   # North mm/m

print(f"G built. Max|G| = {np.abs(G).max():.4f} mm/m")

# Sanity check: uniform 5 cm thrust slip
d_test = G @ (np.ones(N_PATCHES) * 0.05)
print(f"\nUniform 5 cm thrust slip → predicted (E, N) mm:")
for i, sta in enumerate(sta_list):
    print(f"  {sta:8s}: E={d_test[2*i]:+.2f}  N={d_test[2*i+1]:+.2f}")


> **Part III questions:**
> 1. What does each **row** of G represent? Each **column**?
> 2. The predicted east offsets from uniform thrust slip should be negative (westward). Is this what you see?
> 3. Why is the data covariance matrix diagonal here? When would you need a full off-diagonal covariance matrix?
> 4. For 526 patches this builds quickly. What strategies would you use for a 10,000-patch global subduction zone model?


## Part IV: Invert for slip

We solve:

$$\min_{\mathbf{m} \geq 0}\; \left\| \mathbf{W}(\mathbf{G}\mathbf{m} - \mathbf{d}) \right\|^2 + \lambda^2 \|\mathbf{m}\|^2$$

where $\mathbf{W} = \text{diag}(1/\sigma_i)$ and $\mathbf{m} \geq 0$ (non-negativity enforces thrust-only slip).
We solve this with `scipy.optimize.nnls` on the augmented system.


In [ ]:
def solve_sse(G, d_obs, sig_d, lam):
    """Weighted damped NNLS. Returns (m, d_pred, normalized_rms)."""
    W = 1.0 / sig_d
    G_aug = np.vstack([G * W[:, np.newaxis], lam * np.eye(G.shape[1])])
    d_aug = np.concatenate([d_obs * W, np.zeros(G.shape[1])])
    m, _  = nnls(G_aug, d_aug)
    d_pred = G @ m
    nrms   = np.sqrt(np.mean(((d_obs - d_pred) / sig_d)**2))
    return m, d_pred, nrms


# Trial inversion
LAM_TRIAL = 0.05
m_t, d_pred_t, nrms_t = solve_sse(G, d_obs, sig_d, LAM_TRIAL)

print(f"Trial inversion  lambda={LAM_TRIAL}")
print(f"  Normalized RMS misfit: {nrms_t:.3f}  (ideal target = 1)")
print(f"  Max slip:  {m_t.max()*100:.1f} cm")
print(f"  Patches with slip > 1 cm: {(m_t > 0.01).sum()}")

# Observed vs predicted
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, comp, idx in zip(axes, ['East', 'North'], [0, 1]):
    obs_c  = d_obs[idx::2]; pred_c = d_pred_t[idx::2]
    lim = max(np.abs(obs_c).max(), np.abs(pred_c).max()) * 1.2
    ax.errorbar(obs_c, pred_c, xerr=sig_d[idx::2],
                fmt='o', ms=6, color='steelblue', ecolor='0.65', capsize=3)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1)
    for i, sta in enumerate(sta_list):
        ax.text(obs_c[i]+0.1, pred_c[i], sta.upper(), fontsize=7)
    ax.set_xlabel(f'Observed {comp} (mm)'); ax.set_ylabel(f'Predicted {comp} (mm)')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_title(f'{comp} component  (lambda={LAM_TRIAL})')
plt.suptitle('Trial inversion — observed vs predicted', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Map the slip distribution
fig, ax = plt.subplots(figsize=(9, 8))
for code in ('WA', 'OR'):
    ax.plot(US_STATES[code]['lons'], US_STATES[code]['lats'], color='0.35', lw=0.8)

sc = ax.scatter(cent_lon_sse, cent_lat_sse, c=m_t*100,
                cmap='hot_r', s=10, vmin=0, vmax=m_t.max()*100, zorder=3)
plt.colorbar(sc, ax=ax, label='Slip (cm)')

scale = 0.2
for i, sta in enumerate(sta_list):
    v = offsets[sta]
    ax.quiver(v['lon'], v['lat'], v['de']*scale, v['dn']*scale,
              scale=1, scale_units='xy', angles='xy',
              color='steelblue', width=0.004, zorder=5,
              label='Observed' if i == 0 else '_')
    ax.quiver(v['lon'], v['lat'], d_pred_t[2*i]*scale, d_pred_t[2*i+1]*scale,
              scale=1, scale_units='xy', angles='xy',
              color='firebrick', width=0.004, zorder=5,
              label='Predicted' if i == 0 else '_')

ax.quiver(-127.5, 46.3, 5*scale, 0, scale=1, scale_units='xy',
          angles='xy', color='k', width=0.005)
ax.text(-127.5, 46.1, '5 mm', fontsize=8)
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(-129, -121); ax.set_ylim(45.5, 51)
ax.set_aspect(1/np.cos(np.radians(48)))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'2015–2016 Cascadia SSE — slip distribution (lambda={LAM_TRIAL})\n'
             f'Blue = observed, Red = predicted GNSS offsets')
plt.tight_layout(); plt.show()


In [ ]:
# Seismic moment
def tri_area(verts):
    e1 = verts[1] - verts[0]; e2 = verts[2] - verts[0]
    return 0.5 * np.linalg.norm(np.cross(e1, e2))

patch_areas = np.array([tri_area(tris_m[k]) for k in range(N_PATCHES)])
M0 = MU * np.sum(patch_areas * m_t)
Mw = (2/3) * np.log10(M0 * 1e7) - 10.7

print(f"Seismic moment of trial slip distribution:")
print(f"  Total patch area:   {patch_areas.sum()/1e9:.0f} x 10^9 m^2")
print(f"  Area-weighted slip: {np.sum(patch_areas*m_t)/patch_areas.sum()*100:.1f} cm")
print(f"  M0 = {M0:.2e} N*m")
print(f"  Mw = {Mw:.2f}")
print(f"  Published range for 2015-2016 Cascadia SSE: Mw 6.4-6.8 (Bartlow 2020)")


> **Part IV questions:**
> 1. Where does the slip concentrate? At what depth is the peak slip patch?
> 2. Is the fit good? Which stations are most poorly fit, and why?
> 3. What would negative slip represent physically? Why does the non-negativity constraint prevent it?
> 4. Is your Mw consistent with published values? If not, what might explain the discrepancy?


## Part V: L-curve

Sweep $\lambda$ over a log-spaced range. The optimal $\lambda$ is at the **corner of the L**
— the point of maximum curvature where further reducing $\lambda$ rapidly increases model
complexity with little improvement in data fit.


In [ ]:
lambdas = np.logspace(-3, 1, 40)
misfits, norms = [], []

print("Computing L-curve (40 lambda values)...")
for lam in lambdas:
    m, _, nrms = solve_sse(G, d_obs, sig_d, lam)
    misfits.append(nrms)
    norms.append(np.linalg.norm(m))

misfits, norms = np.array(misfits), np.array(norms)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(misfits, norms, c=np.log10(lambdas), cmap='plasma', s=50, zorder=4)
plt.colorbar(sc, ax=ax, label='log10(lambda)')
for lref in [0.01, 0.05, 0.1, 0.5]:
    idx = np.argmin(np.abs(lambdas - lref))
    ax.annotate(f'lam={lref}', xy=(misfits[idx], norms[idx]),
                xytext=(8, 4), textcoords='offset points', fontsize=8, color='0.3')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Normalized RMS misfit')
ax.set_ylabel('Model norm ||m|| (m)')
ax.set_title('L-curve  (corner = optimal lambda)')

ax = axes[1]
ax.semilogx(lambdas, misfits, 'steelblue', lw=2)
ax.set_xlabel('lambda'); ax.set_ylabel('Normalized RMS misfit', color='steelblue')
ax2 = ax.twinx()
ax2.semilogx(lambdas, norms, 'firebrick', lw=2)
ax2.set_ylabel('Model norm (m)', color='firebrick')
ax.set_title('Misfit and model norm vs. lambda')

plt.suptitle('L-curve analysis — 2015-2016 Cascadia SSE inversion', fontsize=11)
plt.tight_layout(); plt.show()

print("\nSet LAM_OPT in the next cell based on the L-curve corner.")


In [ ]:
# ── Task: set your optimal lambda ───────────────────────────────────────────
LAM_OPT = 0.05   # <-- adjust based on your L-curve inspection

m_opt, d_pred_opt, nrms_opt = solve_sse(G, d_obs, sig_d, LAM_OPT)
M0_opt = MU * np.sum(patch_areas * m_opt)
Mw_opt = (2/3) * np.log10(M0_opt * 1e7) - 10.7

print(f"Optimal inversion  lambda={LAM_OPT}")
print(f"  Normalized RMS misfit: {nrms_opt:.3f}")
print(f"  Max slip: {m_opt.max()*100:.1f} cm")
print(f"  Mw: {Mw_opt:.2f}")


> **L-curve questions:**
> 1. Identify the corner of the L-curve. What $\lambda$ does it correspond to?
> 2. Try $\lambda = 0.001$. Describe the slip distribution. What has gone wrong?
> 3. Try $\lambda = 1.0$. What has been sacrificed?
> 4. Should the normalized RMS misfit at the optimal $\lambda$ be close to 1? Is yours? If it is much smaller than 1, what might this say about your uncertainty estimates?
> 5. Name one noise source captured by your uncertainty propagation and one it misses.


## Part VI: Resolution matrix

$$\mathbf{R} = (\mathbf{G}^T\mathbf{C}_d^{-1}\mathbf{G} + \lambda^2\mathbf{I})^{-1}\mathbf{G}^T\mathbf{C}_d^{-1}\mathbf{G}$$

The diagonal of $\mathbf{R}$ is the **self-resolution** of each patch
(1 = perfectly resolved, 0 = completely unresolved).
Rows of $\mathbf{R}$ are resolution kernels — the blurring function for each patch.

We compute $\mathbf{R}$ analytically for the damped least-squares problem
(without the NNLS constraint, which gives a good approximation).


In [ ]:
W_inv = 1.0 / sig_d**2
GtWG  = G.T @ (G * W_inv[:, np.newaxis])
A     = GtWG + LAM_OPT**2 * np.eye(N_PATCHES)
A_inv = np.linalg.inv(A)
R     = A_inv @ GtWG
R_diag = np.diag(R)

print(f"Resolution matrix: {R.shape}")
print(f"Diagonal range: {R_diag.min():.3f} to {R_diag.max():.3f}")
print(f"Patches with R > 0.5 (well-resolved): {(R_diag > 0.5).sum()}")
print(f"Patches with R < 0.1 (poorly resolved): {(R_diag < 0.1).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for code in ('WA', 'OR'):
    ax.plot(US_STATES[code]['lons'], US_STATES[code]['lats'], color='0.35', lw=0.8)
sc = ax.scatter(cent_lon_sse, cent_lat_sse, c=R_diag,
                cmap='viridis', s=10, vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, label='Self-resolution (diagonal of R)')
for sta, v in offsets.items():
    ax.plot(v['lon'], v['lat'], 'k^', ms=5, zorder=5)
ax.set_xlim(-129, -121); ax.set_ylim(45.5, 51)
ax.set_aspect(1/np.cos(np.radians(48)))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Resolution diagonal  (lambda={LAM_OPT})')

ax = axes[1]
ax.scatter(-cent_dep_sse, R_diag, c=cent_lat_sse, cmap='RdYlBu', s=15, alpha=0.7)
ax.axhline(0.5, color='k', ls='--', lw=1, label='R = 0.5')
ax.set_xlabel('Depth (km)'); ax.set_ylabel('Self-resolution')
ax.set_title('Resolution vs. depth'); ax.legend(fontsize=9)

plt.suptitle('Resolution analysis — 2015-2016 Cascadia SSE', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Resolution kernels for best and worst resolved patches
idx_best  = np.argmax(R_diag)
idx_worst = np.argmin(R_diag)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, idx, label in zip(axes,
        [idx_best, idx_worst], ['Best resolved', 'Worst resolved']):
    for code in ('WA', 'OR'):
        ax.plot(US_STATES[code]['lons'], US_STATES[code]['lats'], color='0.35', lw=0.8)
    kernel = R[idx, :]
    vmax = np.abs(kernel).max()
    sc = ax.scatter(cent_lon_sse, cent_lat_sse, c=kernel,
                    cmap='RdBu_r', s=10, vmin=-vmax, vmax=vmax)
    plt.colorbar(sc, ax=ax, label='Kernel weight')
    ax.scatter(cent_lon_sse[idx], cent_lat_sse[idx],
               s=120, c='k', marker='*', zorder=6,
               label=f'depth={-cent_dep_sse[idx]:.0f} km, R={R_diag[idx]:.2f}')
    for sta, v in offsets.items():
        ax.plot(v['lon'], v['lat'], 'k^', ms=5, zorder=5)
    ax.set_xlim(-129, -121); ax.set_ylim(45.5, 51)
    ax.set_aspect(1/np.cos(np.radians(48)))
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(f'{label} (star)\ndepth={-cent_dep_sse[idx]:.0f} km')
    ax.legend(fontsize=8)

plt.suptitle('Resolution kernels — rows of R for two patches', fontsize=11)
plt.tight_layout(); plt.show()


> **Part VI questions:**
> 1. Where are the best- and worst-resolved patches? Explain in terms of the station distribution.
> 2. Is the best-resolved kernel compact or spread? What would a perfect kernel look like?
> 3. What does a broad kernel for the worst-resolved patch mean physically — what are you actually measuring when you report slip there?
> 4. How does resolution vary with depth? Is this consistent with what you learned in Lab 10?
> 5. Do the high-slip patches from Part IV lie in well- or poorly-resolved regions? How does this affect your confidence in the result?


## Synthesis

Write 3–5 sentences for each question.

> **1. The slip distribution**  
> Describe where the slip concentrates and at what depth. Is this consistent with published Cascadia SSE locations (Bartlow 2020)? What are you most and least confident in?

> **2. Regularization**  
> Explain what $\lambda$ controls and why it cannot be zero. How did you choose your optimal $\lambda$? What happens at $\lambda \times 10$ and $\lambda / 10$?

> **3. Data coverage and resolution**  
> The station network is concentrated on Vancouver Island and the Washington coast. How does this limit resolution? Where would you add three new GNSS stations to most improve the inversion, and why?

> **4. Connecting to the seismic cycle**  
> Cascadia SSEs recur every 12–14 months; each releases ~1–5 cm of slip; the long-term convergence rate is ~40 mm/yr. What fraction of convergence is released as slow slip? How does your Mw estimate connect to the moment budget in Lab 4?

### References

- Bartlow, N. M. (2020). A long-term view of episodic tremor and slip in the Pacific Northwest. *GRL*, 47. https://doi.org/10.1029/2019GL085303
- McCrory et al. (2012). Juan de Fuca slab geometry. *JGR*, 117. https://doi.org/10.1029/2012JB009407
- Meade, B. J. (2007). Triangular dislocation elements. *Computers & Geosciences*, 33, 1064–1075.
- Li et al. (2018). Cascadia locking state. *JGR*, 123. https://doi.org/10.1029/2018JB015620
- CRESCENT GNSS dataset: https://zenodo.org/records/20518159


## Summary

- **SSE offsets** are extracted by fitting trend + seasonal models (as in Lab 2) and differencing pre/post-event windows. The per-station uncertainties $\sigma_{offset}$ propagated from daily sigmas populate the diagonal of the data covariance matrix $\mathbf{C}_d$ used to weight the inversion.

- The **McCrory et al. mesh** (526 patches, 46–50°N, 8–35 km) provides a physically realistic triangulated fault geometry for northern Cascadia. Converting from lon/lat/km to local Cartesian meters is required by cutde.

- The **Green's function matrix** $\mathbf{G}$ ($M \times N$) maps unit thrust slip on each patch to surface displacement. The system is strongly underdetermined ($M \ll N$), so regularization is essential.

- **Damped non-negative least squares** combines a weighted data-fit penalty, a model-norm penalty ($\lambda^2\|\mathbf{m}\|^2$), and non-negativity to enforce thrust-only slip consistent with SSE physics.

- The **L-curve** identifies the optimal $\lambda$ at the corner between data-fitting and model-smoothing regimes. The normalized RMS misfit at the optimal solution should be close to 1.

- The **resolution matrix** $\mathbf{R}$ shows that patches near the coastal station network are well resolved, while offshore and deep patches are poorly constrained — a fundamental limitation of land-based geodetic networks.

- The recovered slip and equivalent Mw connect directly to the **moment budget** in Lab 4 and the **coupling models** in the seismic cycle lecture — the same physical process viewed from multiple timescales and datasets.
